# P2 Colab GPU smoke
런타임 → 런타임 유형 변경 → GPU를 선택한 뒤 실행하세요. 본실험 학습을 실행하지 않습니다. 로컬에서 생성한 `p2_colab_bundle_v1.zip`을 업로드합니다. CPU corpus/fixture를 GPU에서 생성하지 않습니다.

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, zipfile, os, sys
uploaded = files.upload()
archive = Path('p2_colab_bundle_v1.zip').resolve()
assert hashlib.sha256(archive.read_bytes()).hexdigest() == '91e1d91d24349e6390600c0bb85e33cc30eb7cb8d5be3747430189ba7ac7251a', 'Bundle checksum mismatch'
root = Path('/content/boolean_interp')
root.mkdir(exist_ok=False)
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        assert not Path(name).is_absolute() and '..' not in Path(name).parts
    z.extractall(root)
os.chdir(root)
print('Verified bundle:', hashlib.sha256(archive.resolve().read_bytes()).hexdigest())


In [ ]:
# p2_colab_bundle_v1.zip에는 experiment_v1/results/run_registry.csv가 빠져 있고
# tests/test_p0_config.py가 이 파일을 읽는다. 여기서 따로 업로드한다(1.2KB).
# registry는 단계마다 갱신되므로 hash를 고정하지 않고 실제 값을 기록한다.
from google.colab import files
import hashlib
files.upload()                      # experiment_v1/results/run_registry.csv 선택
data = Path('run_registry.csv').read_bytes()
dest = root / 'experiment_v1/results/run_registry.csv'
dest.parent.mkdir(parents=True, exist_ok=True)
dest.write_bytes(data)
Path('run_registry.csv').unlink()
print('placed', dest, len(data), 'bytes, sha256', hashlib.sha256(data).hexdigest())


In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-interp.txt"], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), "A CUDA GPU runtime is required"
subprocess.run([sys.executable, "-m", "pytest", "tests", "-q"], check=True)
subprocess.run([sys.executable, "-m", "interp.smoke", "--device", "cuda", "--output", "experiment_v1/smoke/colab_gpu_01"], check=True)


In [ ]:
import json, shutil
result_dir = root / "experiment_v1/smoke/colab_gpu_01"
result = json.loads((result_dir / "smoke.json").read_text())
assert result["status"] == "passed" and result["scope"] == "cuda"
print(json.dumps({"status": result["status"], "environment": result["environment"], "elapsed_seconds": result["elapsed_seconds"]}, indent=2))
archive_out = shutil.make_archive("/content/p2_colab_gpu_evidence", "zip", result_dir)
files.download(archive_out)
